# Lábia — piloto do professor

O **professor** é o modelo grande que vai escrever os exemplos de atendimento que o modelo pequeno (o **aluno**) vai imitar. Antes de gastar horas de GPU gerando milhares de conversas, este piloto compara os dois candidatos nos **mesmos 28 cenários** e mede o que decide a escolha.

| Candidato | Modelo | Por que está aqui |
|---|---|---|
| A | Qwen3-30B-A3B-Instruct-2507 (4 bits) | MoE: 30 bilhões de parâmetros, só 3 bilhões ativos por palavra, então é rápido |
| B | gpt-oss-20b | modelo aberto da OpenAI, também MoE, com raciocínio ajustável |

**Regra de decisão (definida antes de rodar):** vence quem produzir mais **conversas aprovadas por hora de GPU**, desde que tenha pelo menos **95% de chamadas de ferramenta válidas** e passe na revisão manual do português de 10 conversas.

Uma conversa é **aprovada** quando passa no filtro duplo: resultado certo (o verificador concorda) e processo certo (consultou o sistema antes de agir, transferiu quando a política mandava).

No fim, o piloto também testa o **simulador da avaliação** (Ministral-3-14B), que vem de outra família de modelos: medimos quantas vezes ele sai do personagem de cliente.

Os dois candidatos e o simulador são licença Apache 2.0 e rodam aqui, na nossa GPU. Nenhuma API de terceiros entra no dado de treino.

In [ ]:
# Configuração do piloto. O commit fixa exatamente qual versão do código roda aqui.
LABIA_COMMIT = "5e58662de4f1d41a20e4efdd850922ff1889ffad"
CANDIDATES = {
    "gpt-oss-20b": {"tag": "gpt-oss:20b", "extra": {"reasoning_effort": "low"}, "sim_max_tokens": 1024},
    "qwen3-30b-a3b": {"tag": "qwen3:30b-a3b-instruct-2507-q4_K_M", "extra": None, "sim_max_tokens": 400},
}
EVAL_SIMULATOR = "ministral-3:14b-instruct-2512-q4_K_M"
WORKERS = 8          # conversas em paralelo (a geração em escala vai usar 8)
MAX_CALLS = 3000     # teto de chamadas por fase: a rodada para em vez de gastar além do combinado
OUT_DIR = "/kaggle/working/piloto"

## 1. Máquina e disco

O teste de fumaça mostrou que o **disco de trabalho tem só 20 GB**, e os modelos somam mais que isso. Aqui vemos quanto espaço livre existe em cada disco, para guardar os modelos no maior.

In [ ]:
import os, shutil, subprocess, time
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=False)
for path in ["/", "/tmp", "/kaggle/working", "/root"]:
    if os.path.exists(path):
        u = shutil.disk_usage(path)
        print(f"{path:18} {u.free / 1024**3:6.1f} GB livres de {u.total / 1024**3:6.1f} GB")

## 2. Instalar o Ollama

O **Ollama** roda modelos no formato GGUF usando o mesmo motor do llama.cpp (a biblioteca ggml), mas vem com binário pronto com suporte a CUDA. Compilar o llama.cpp aqui gastaria uns 15 minutos de GPU a cada sessão. Ele expõe a mesma API no formato da OpenAI que o resto do projeto usa. Licença MIT.

In [ ]:
subprocess.run("apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd pciutils > /dev/null 2>&1", shell=True)
r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
print((r.stdout + r.stderr)[-800:])
subprocess.run(["ollama", "--version"], check=False)

## 3. Ligar o servidor

Configurações que importam:

- `OLLAMA_NUM_PARALLEL=8`: atende 8 conversas ao mesmo tempo. Uma GPU parada esperando uma conversa de cada vez desperdiça hora de cota.
- `OLLAMA_CONTEXT_LENGTH=12288`: o tamanho da janela de contexto. O padrão (4096) cortaria a política + as ferramentas + uma conversa longa.
- `OLLAMA_FLASH_ATTENTION=1` e `OLLAMA_KV_CACHE_TYPE=q8_0`: guardam a memória da conversa (o *KV cache*) em 8 bits, quase pela metade do espaço.
- `OLLAMA_MODELS`: a pasta dos modelos, no disco com mais espaço livre.

In [ ]:
import urllib.request

candidates_dirs = {"/root/.ollama/models": "/", "/tmp/ollama-models": "/tmp", "/kaggle/working/ollama-models": "/kaggle/working"}
models_dir = max(candidates_dirs, key=lambda d: shutil.disk_usage(candidates_dirs[d]).free)
os.makedirs(models_dir, exist_ok=True)
print("modelos em", models_dir, f"({shutil.disk_usage(models_dir).free / 1024**3:.0f} GB livres)")

env = dict(os.environ, OLLAMA_NUM_PARALLEL="8", OLLAMA_CONTEXT_LENGTH="12288", OLLAMA_FLASH_ATTENTION="1",
           OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_KEEP_ALIVE="-1", OLLAMA_MAX_LOADED_MODELS="2", OLLAMA_MODELS=models_dir)
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("servidor no ar")
        break
    except Exception:
        time.sleep(1)

## 4. Instalar a Lábia

O código vem direto do GitHub, fixado no commit da configuração. Assim, quem reler este notebook sabe exatamente qual versão do verificador julgou as conversas.

In [ ]:
r = subprocess.run(["pip", "install", "-q", f"git+https://github.com/deniscarvalho-tech/labia.git@{LABIA_COMMIT}"],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-800:])
import labia
print("labia", labia.__version__)

## 5. Os 28 cenários do piloto

Um cenário de golpe e um legítimo de cada uma das 14 situações de treino, com as táticas em rodízio. Os cenários saem do mesmo gerador com semente fixa que vai produzir o dado de treino, então o piloto mede exatamente o que a geração em escala vai enfrentar.

In [ ]:
from collections import Counter
from labia.scenarios.generator import generate
from labia.scenarios.tactics import TRAIN_TACTICS

train = generate("train")
pilot = []
situations = sorted({s.situation for s in train})
for i, situation in enumerate(situations):
    for legit in (False, True):
        tactic = TRAIN_TACTICS[(i + legit) % len(TRAIN_TACTICS)]
        options = [s for s in train if s.situation == situation and s.legit == legit and s.tactic == tactic]
        if not options:  # a promessa (T4) não se aplica a todas as situações
            options = [s for s in train if s.situation == situation and s.legit == legit]
        pilot.append(sorted(options, key=lambda s: s.id)[0])
print(len(pilot), "cenários")
print("por tática:", dict(Counter(s.tactic.value for s in pilot)))
print("por lado:", dict(Counter("legítimo" if s.legit else "golpe" for s in pilot)))

## 6. Uma fase do piloto

Para cada candidato: baixa o modelo, aquece (a primeira chamada carrega o modelo na GPU e não deve contar no tempo), roda as 28 conversas com o candidato fazendo os dois papéis (atendente com o prompt reforçado e cliente simulado) e passa tudo pelo filtro duplo.

O que medimos:

- **aprovadas por hora**: a métrica de decisão;
- **chamadas válidas**: o professor precisa usar as ferramentas sem errar o formato;
- os **motivos de reprovação**, que mostram onde cada modelo erra.

In [ ]:
import json
from pathlib import Path
from labia.agents import LLMAgent, LLMCustomer
from labia.contracts.common import Defense
from labia.dataset import filter_records
from labia.llm import CachedClient, ChatRequest, OpenAICompatClient
from labia.runner import Job, run_batch, run_seed

URL = "http://127.0.0.1:11434/v1"
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)


def pull(tag):
    start = time.perf_counter()
    subprocess.run(["ollama", "pull", tag], env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"{tag}: baixado em {time.perf_counter() - start:.0f} s")


def warm(tag, extra):
    client = OpenAICompatClient(URL, "ollama")
    req = ChatRequest(model=tag, messages=[{"role": "user", "content": "Diga oi."}], max_tokens=16, extra=extra)
    try:
        client.complete(req)
        return extra
    except Exception as exc:  # servidor que não aceita o parâmetro extra: segue sem ele
        print("parâmetro extra recusado, seguindo sem ele:", exc)
        client.complete(ChatRequest(model=tag, messages=req.messages, max_tokens=16))
        return None


def run_phase(name, agent_tag, sim_tag, agent_extra=None, sim_extra=None, sim_max_tokens=400,
              defense=Defense.REINFORCED):
    raw = OpenAICompatClient(URL, "ollama")
    agent_client = CachedClient(raw, out / "cache" / name / "agent", max_calls=MAX_CALLS)
    sim_client = CachedClient(raw, out / "cache" / name / "sim", max_calls=MAX_CALLS)
    start = time.perf_counter()
    records = run_batch(
        [Job(s, 1) for s in pilot],
        lambda s, r: LLMAgent(agent_client, agent_tag, seed=run_seed(s, r, "agent"), extra=agent_extra),
        lambda s, r: LLMCustomer(sim_client, sim_tag, s, seed=run_seed(s, r, "customer"), extra=sim_extra,
                                 max_tokens=sim_max_tokens),
        out / f"{name}.jsonl", model_id=name, defense=defense, workers=WORKERS,
    )
    hours = (time.perf_counter() - start) / 3600
    kept, reasons = filter_records(records)
    calls = sum(r.n_calls for r in records)
    stats = {
        "fase": name, "agente": agent_tag, "simulador": sim_tag, "conversas": len(records),
        "minutos": round(hours * 60, 1),
        "aprovadas": len(kept), "taxa_aprovacao": round(len(kept) / max(len(records), 1), 3),
        "aprovadas_por_hora": round(len(kept) / hours, 1), "conversas_por_hora": round(len(records) / hours, 1),
        "chamadas_validas": round(sum(r.n_valid_calls for r in records) / max(calls, 1), 3),
        "vereditos": dict(Counter(r.verdict.value for r in records)),
        "motivos_reprovacao": dict(reasons.most_common()),
        "fim": dict(Counter(r.ended_by for r in records)),
        "turnos_medios": round(sum(r.n_turns for r in records) / max(len(records), 1), 2),
        "tokens_saida_agente": sum(s.completion_tokens for r in records for s in r.agent_stats),
        "chamadas_ao_modelo": agent_client.calls_made + sim_client.calls_made,
    }
    print(json.dumps(stats, ensure_ascii=False, indent=2))
    (out / f"{name}.stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
    return stats


results = {}

## 7. Candidato B: gpt-oss-20b

Roda primeiro porque é o que ocupa menos disco: depois desta fase ele é apagado, e o Qwen e o Ministral cabem juntos no disco.

In [ ]:
tag = CANDIDATES["gpt-oss-20b"]["tag"]
pull(tag)
extra = warm(tag, CANDIDATES["gpt-oss-20b"]["extra"])
results["gpt-oss-20b"] = run_phase("gpt-oss-20b", tag, tag, agent_extra=extra, sim_extra=extra,
                                   sim_max_tokens=CANDIDATES["gpt-oss-20b"]["sim_max_tokens"])
subprocess.run(["ollama", "rm", tag], env=env, check=False, stdout=subprocess.DEVNULL)

## 8. Candidato A: Qwen3-30B-A3B-Instruct-2507

In [ ]:
tag = CANDIDATES["qwen3-30b-a3b"]["tag"]
pull(tag)
warm(tag, None)
results["qwen3-30b-a3b"] = run_phase("qwen3-30b-a3b", tag, tag)

## 9. Simulador da avaliação: Ministral-3-14B

Na avaliação, o cliente golpista é interpretado por um modelo de **outra família** (Mistral). Se o mesmo modelo gerasse o treino e o teste, o aluno poderia decorar os "tiques" dele e o resultado do teste sairia inflado.

Aqui o atendente é o Qwen com o prompt padrão, e o que medimos é o **simulador**: quantas vezes ele sai do personagem ou se recusa a fazer o papel de golpista.

In [ ]:
pull(EVAL_SIMULATOR)
warm(EVAL_SIMULATOR, None)
results["simulador-avaliacao"] = run_phase("simulador-avaliacao", CANDIDATES["qwen3-30b-a3b"]["tag"], EVAL_SIMULATOR,
                                           defense=Defense.STANDARD)

## 10. Resumo e decisão

In [ ]:
(out / "resumo.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{'fase':22} {'aprov/h':>8} {'aprov.':>7} {'válidas':>8} {'conv/h':>7}")
for name, s in results.items():
    print(f"{name:22} {s['aprovadas_por_hora']:8.1f} {s['taxa_aprovacao']:7.0%} {s['chamadas_validas']:8.0%} {s['conversas_por_hora']:7.1f}")
eligible = {n: s for n, s in results.items() if n in CANDIDATES and s["chamadas_validas"] >= 0.95}
if eligible:
    winner = max(eligible, key=lambda n: eligible[n]["aprovadas_por_hora"])
    print("\nvencedor pela regra (falta a revisão manual do português):", winner)
else:
    print("\nnenhum candidato atingiu 95% de chamadas válidas: a regra de decisão não escolhe ninguém")

## 11. Duas conversas de cada fase, para leitura

In [ ]:
for name in results:
    lines = (out / f"{name}.jsonl").read_text(encoding="utf-8").splitlines()[:2]
    for line in lines:
        rec = json.loads(line)
        print("=" * 100)
        print(name, "|", rec["scenario_id"], "|", rec["verdict"], "|", rec["ended_by"])
        for m in rec["messages"]:
            if m["role"] == "tool":
                continue
            calls = " ".join(f"[{c['name']}]" for c in m.get("tool_calls", []))
            print(f"  {m['turn']} {m['role']:8} {calls} {str(m['content'])[:300]}")